In [ ]:
import subprocess, sys, os, shutil
os.environ.setdefault('HF_HUB_ENABLE_HF_TRANSFER','0')
subprocess.run([sys.executable,'-m','pip','install','-q','uv'],check=False)
UV=shutil.which('uv') or 'uv'
def run(cmd,phase):
    p=subprocess.run(cmd,capture_output=True,text=True)
    print(phase,'-> exit',p.returncode)
    if p.returncode!=0:
        print(p.stdout[-1200:]); print(p.stderr[-1200:]); raise RuntimeError(phase+' failed')
R=['torch>=2.8.0','triton>=3.4.0','transformers==4.56.2','peft==0.20.0','trl==0.22.2',
   'datasets==5.0.1','accelerate==1.15.0','bitsandbytes==0.50.2','openai-harmony==0.0.8']
run([UV,'pip','install','--system','--python',sys.executable,'--no-cache-dir',*R],'resolver')
run([UV,'pip','install','--system','--python',sys.executable,'--no-cache-dir','--no-deps','--upgrade',
     'unsloth==2026.9.4','unsloth_zoo==2026.9.3'],'frozen-no-deps')
print('install complete')


In [ ]:
# GHARIBO EXP-002 — V1 QUALIFICATION INFERENCE (20 FROZEN QUESTIONS, INFERENCE ONLY)
#
# Same inference path as the GREEN benchmark. Gold is ABSENT from this payload;
# scoring happens locally.
#
# ROOT CAUSE OF THE >900s/ITEM INCIDENT (fixed here):
#   1. NO stopping criterion. `generate()` was called with max_new_tokens=3072 and
#      no Harmony terminator handling, so it ran to the FULL 3072-token ceiling
#      every time instead of stopping at <|return|>. That alone is ~20-40x the
#      ~600-1100 tokens a gold answer actually needs.
#   2. TWO-GPU SHARDING. Unsloth split weights 5.68 GiB per T4 with `lm_head` on
#      cuda:1, so every generated token crossed the PCIe bus for the LM head.
#      The model fits on ONE T4, so sharding only cost time.
#   3. torch.no_grad() instead of torch.inference_mode().
#
# These are RUNTIME fixes only: adapter, base model, prompts, template and
# generation semantics (greedy) are unchanged.
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'   # FIX 2: single GPU, no cross-device lm_head
import json, time, torch, pathlib, re

PROMPTS = json.loads(r'''[{"item_id": "922a508d37f1e36c8a294d20a5ba55705fe7408d500e3b9e0696bf0e72959eb1", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=PRODUCT_FAMILY, externalKey=family:suprema:corestation) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"PRODUCT_FAMILY\",\n    \"externalKey\": \"family:suprema:corestation\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T21:17:52+03:00\",\n    \"payload\": {\n      \"manufacturer\": \"Suprema\",\n      \"brand\": \"Suprema\",\n      \"family\": \"CoreStation\",\n      \"domain\": \"Security Systems\",\n      \"category\": \"Access Controller\",\n      \"lifecycle\": \"CURRENT\",\n      \"name\": \"CoreStation\",\n      \"description\": \"Suprema current product line listed in the official hardware selector.\",\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:suprema-selector2\",\n        \"sourceUrl\": \"https://supremainc.com/en/hardware/product_selector.asp?iCTG_No=&iPage=2&iPageSize=20\",\n        \"sourceType\": \"MANUFACTURER_OFFICIAL\",\n        \"observedAt\": \"2026-09-04T21:17:52+03:00\",\n        \"claim\": \"Official source enumerates or identifies the CoreStation product family/series.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "d0c064337d2035d4c048dd1d7ea0e176247e0698dea3830fa56fe7b4dc6cd3bc", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=RELATION, externalKey=relation:branded_by:b1a70f49b9b5e2014a54) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"RELATION\",\n    \"externalKey\": \"relation:branded_by:b1a70f49b9b5e2014a54\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T22:36:04+03:00\",\n    \"payload\": {\n      \"relationType\": \"BRANDED_BY\",\n      \"sourceExternalKey\": \"model:tp-link:omada-agile-switches:es208g\",\n      \"targetExternalKey\": \"brand:omada\",\n      \"confidence\": \"HIGH\",\n      \"notes\": \"ES208G is marketed under the Omada brand.\"\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:batch003:model:3d397736b690cba4b260632b\",\n        \"sourceUrl\": \"https://www.omadanetworks.com/us/business-networking/all-omada-switch/\",\n        \"sourceType\": \"MANUFACTURER_OFFICIAL\",\n        \"observedAt\": \"2026-09-04T22:36:04+03:00\",\n        \"claim\": \"ES208G is marketed under the Omada brand.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "73a65e798efda5848ed75a9c9364f68550b2d41c2e5af65e250ccf608efa8680", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=RELATION, externalKey=relation:component_of:812ff5faad9c49f9470f) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"RELATION\",\n    \"externalKey\": \"relation:component_of:812ff5faad9c49f9470f\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T22:36:04+03:00\",\n    \"payload\": {\n      \"relationType\": \"COMPONENT_OF\",\n      \"sourceExternalKey\": \"model:hikvision:ds-81-recorder-family:ds-8116hqhi-f8-n\",\n      \"targetExternalKey\": \"system:security:nvr-ip-surveillance\",\n      \"confidence\": \"HIGH\",\n      \"notes\": \"DS-8116HQHI-F8/N is commercially relevant as a component of NVR-based IP Surveillance System.\"\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:batch003:model:6cb05b3b487de2a92e5ccdb5\",\n        \"sourceUrl\": \"https://opensource.hikvision.com/Home/List?id=12&page=3\",\n        \"sourceType\": \"MANUFACTURER_OFFICIAL\",\n        \"observedAt\": \"2026-09-04T22:36:04+03:00\",\n        \"claim\": \"DS-8116HQHI-F8/N is commercially relevant as a component of NVR-based IP Surveillance System.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "b529efa8f6332bf7473bb21f002038218f9eb01a5e5cebb5837fc0487a657752", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=RELATION, externalKey=relation:security-registry:parent_of:593fd4fb7bd0b88c9863) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"RELATION\",\n    \"externalKey\": \"relation:security-registry:parent_of:593fd4fb7bd0b88c9863\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T23:06:59+03:00\",\n    \"payload\": {\n      \"relationType\": \"PARENT_OF\",\n      \"sourceExternalKey\": \"system:security:networked-access-control\",\n      \"targetExternalKey\": \"system:security:vehicle-gate-access-control\",\n      \"confidence\": \"HIGH\",\n      \"notes\": \"Networked Access Control System is the parent architecture for Vehicle Gate / Barrier Access Control System.\"\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:security-registry:34647fb236ed69a563778492\",\n        \"sourceUrl\": \"https://www.genetec.com/products/unified-security/synergis\",\n        \"sourceType\": \"OFFICIAL_OR_STANDARDS_BODY\",\n        \"observedAt\": \"2026-09-04T23:06:59+03:00\",\n        \"claim\": \"Networked Access Control System is the parent architecture for Vehicle Gate / Barrier Access Control System.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "390eb3c17592abd6c65041718f0155a44e51d7718523cd0d8b5996330374d33b", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=PRODUCT_MODEL, externalKey=model:i-pro:i-pro-s-series:wv-s32402-f2l) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"PRODUCT_MODEL\",\n    \"externalKey\": \"model:i-pro:i-pro-s-series:wv-s32402-f2l\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T22:36:04+03:00\",\n    \"payload\": {\n      \"manufacturer\": \"i-PRO\",\n      \"brand\": \"i-PRO\",\n      \"family\": \"i-PRO S Series\",\n      \"series\": \"i-PRO S Series\",\n      \"model\": \"WV-S32402-F2L\",\n      \"modelNumber\": \"WV-S32402-F2L\",\n      \"mpn\": null,\n      \"sku\": null,\n      \"gtin\": null,\n      \"officialProductTitle\": \"WV-S32402-F2L\",\n      \"domain\": \"Security Systems\",\n      \"category\": \"Network Camera\",\n      \"subcategory\": null,\n      \"productRole\": \"Network camera\",\n      \"systemRole\": \"IP Video Surveillance System\",\n      \"lifecycle\": \"CURRENT\",\n      \"name\": \"WV-S32402-F2L\",\n      \"description\": null,\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [\n      {\n        \"type\": \"MODEL_NUMBER\",\n        \"value\": \"WV-S32402-F2L\",\n        \"issuer\": \"i-PRO\"\n      }\n    ],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:batch003:model:63c47e91eb720f4dc95c8f6c\",\n        \"sourceUrl\": \"https://i-pro.com/products_and_solutions/en/surveillance/products-list?series=S-series\",\n        \"sourceType\": \"MANUFACTURER_OFFICIAL\",\n        \"observedAt\": \"2026-09-04T22:36:04+03:00\",\n        \"claim\": \"Official source supports product/model identity WV-S32402-F2L.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "ad007a20fbae8254204badeda85bbff787e59d2b6292e886bf8d5eb89a7ebfca", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=SERVICE, externalKey=service:security:commissioning) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"SERVICE\",\n    \"externalKey\": \"service:security:commissioning\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T23:06:59+03:00\",\n    \"payload\": {\n      \"name\": \"Security System Commissioning\",\n      \"domain\": \"Security Systems\",\n      \"registryLayer\": \"SERVICE\",\n      \"serviceClass\": \"COMMISSIONING\",\n      \"purpose\": \"Test and bring a newly deployed security system into operational service.\",\n      \"typicalActivities\": [\n        \"Perform commissioning tests\",\n        \"Validate subsystem operation\",\n        \"Address deployment issues\"\n      ],\n      \"typicalDeliverables\": [\n        \"Commissioned system / findings\"\n      ],\n      \"applicableSystemGroups\": [\n        \"ALL_SECURITY_SYSTEMS\"\n      ],\n      \"deliveryModes\": [\n        \"On-site\",\n        \"Remote\"\n      ],\n      \"recurring\": false,\n      \"prerequisites\": [],\n      \"vendorNeutralRegistryService\": true,\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [\n      {\n        \"code\": \"service_class\",\n        \"name\": \"Service class\",\n        \"value\": \"COMMISSIONING\",\n        \"unit\": null,\n        \"group\": \"Registry\",\n        \"dataType\": \"string\"\n      },\n      {\n        \"code\": \"registry_group\",\n        \"name\": \"Registry group\",\n        \"value\": \"category:security:services:deployment\",\n        \"unit\": null,\n        \"group\": \"Registry\",\n        \"dataType\": \"string\"\n      },\n      {\n        \"code\": \"recurring\",\n        \"name\": \"Recurring service\",\n        \"value\": false,\n        \"unit\": null,\n        \"group\": \"Registry\",\n        \"dataType\": \"boolean\"\n      }\n    ],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:security-registry:db47fcab0a4773bf80d8a67f\",\n        \"sourceUrl\": \"https://resources.genetec.com/i/1319812-genetec-professional-services\",\n        \"sourceType\": \"OFFICIAL_OR_STANDARDS_BODY\",\n        \"observedAt\": \"2026-09-04T23:06:59+03:00\",\n        \"claim\": \"Source supports the commercial/professional service activity represented by Security System Commissioning.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "b12a115a64cb1462d3d58c8eb24228ced20fa1dab238ab33e0931e0d98ea4f06", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=RELATION, externalKey=relation:belongs_to_category:87008cccd9f17d93) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"RELATION\",\n    \"externalKey\": \"relation:belongs_to_category:87008cccd9f17d93\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T21:17:52+03:00\",\n    \"payload\": {\n      \"relationType\": \"BELONGS_TO_CATEGORY\",\n      \"sourceExternalKey\": \"model:suprema:door-interface:di-24\",\n      \"targetExternalKey\": \"category:security:access-io-module\",\n      \"confidence\": \"HIGH\",\n      \"notes\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:suprema-selector3\",\n        \"sourceUrl\": \"https://supremainc.com/en/hardware/product_selector.asp?iCTG_No=&iPage=3&iPageSize=20\",\n        \"sourceType\": \"MANUFACTURER_OFFICIAL\",\n        \"observedAt\": \"2026-09-04T21:17:52+03:00\",\n        \"claim\": \"Official source supports BELONGS_TO_CATEGORY relation.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "e44650f2658617f7cd5d8cf158dcb79526761102b2c8455e07e9f110f3b6ff5b", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=RELATION, externalKey=relation:security-registry:parent_of:7733b07a0fc011272e6b) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"RELATION\",\n    \"externalKey\": \"relation:security-registry:parent_of:7733b07a0fc011272e6b\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T23:06:59+03:00\",\n    \"payload\": {\n      \"relationType\": \"PARENT_OF\",\n      \"sourceExternalKey\": \"system:security:ip-video-surveillance\",\n      \"targetExternalKey\": \"system:security:onvif-interoperable-ip-video\",\n      \"confidence\": \"HIGH\",\n      \"notes\": \"IP Video Surveillance System is the parent architecture for ONVIF-interoperable IP Video System.\"\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:security-registry:79067ab5df616882df208380\",\n        \"sourceUrl\": \"https://www.onvif.org/profiles/\",\n        \"sourceType\": \"OFFICIAL_OR_STANDARDS_BODY\",\n        \"observedAt\": \"2026-09-04T23:06:59+03:00\",\n        \"claim\": \"IP Video Surveillance System is the parent architecture for ONVIF-interoperable IP Video System.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "4994c9da66ae344decb951076ba3c90702a5ef90b9a380062f6e51cd1e10ddee", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=SYSTEM, externalKey=system:security:wireless-lock-access-control) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"SYSTEM\",\n    \"externalKey\": \"system:security:wireless-lock-access-control\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T23:06:59+03:00\",\n    \"payload\": {\n      \"name\": \"Wireless Electronic-lock Access Control System\",\n      \"domain\": \"Security Systems\",\n      \"registryLayer\": \"SYSTEM\",\n      \"architectureKind\": \"WIRELESS_LOCK_ARCHITECTURE\",\n      \"purpose\": \"Networked access architecture integrating supported wireless electronic locks through gateways/controllers.\",\n      \"lifecycle\": \"CURRENT\",\n      \"signalTransport\": null,\n      \"managementModel\": null,\n      \"recordingModel\": null,\n      \"typicalComponents\": [\n        \"Wireless electronic locks\",\n        \"Wireless/IP gateway or supported controller\",\n        \"Access software\"\n      ],\n      \"requiredComponentsOrConditions\": [\n        \"Explicit lock/gateway/controller/software compatibility\"\n      ],\n      \"optionalComponents\": [],\n      \"protocols\": [],\n      \"dependencies\": [],\n      \"topology\": null,\n      \"capacityConsiderations\": [],\n      \"designConstraints\": [],\n      \"installationConsiderations\": [],\n      \"licensingServiceDependencies\": [],\n      \"interoperabilityConstraints\": [],\n      \"standardsMaturity\": null,\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [\n      {\n        \"code\": \"architecture_kind\",\n        \"name\": \"Architecture kind\",\n        \"value\": \"WIRELESS_LOCK_ARCHITECTURE\",\n        \"unit\": null,\n        \"group\": \"Registry\",\n        \"dataType\": \"string\"\n      },\n      {\n        \"code\": \"registry_group\",\n        \"name\": \"Registry group\",\n        \"value\": \"category:security:systems:access-control\",\n        \"unit\": null,\n        \"group\": \"Registry\",\n        \"dataType\": \"string\"\n      },\n      {\n        \"code\": \"lifecycle\",\n        \"name\": \"Architecture lifecycle\",\n        \"value\": \"CURRENT\",\n        \"unit\": null,\n        \"group\": \"Registry\",\n        \"dataType\": \"string\"\n      }\n    ],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:security-registry:87c61e3c4feaa49a0f4c2047\",\n        \"sourceUrl\": \"https://synergis-cloudlink-help.genetec.com/EN/EN/SSW/T_SSW_Enrolling_AllegionSchlageIPLocks.html\",\n        \"sourceType\": \"OFFICIAL_OR_STANDARDS_BODY\",\n        \"observedAt\": \"2026-09-04T23:06:59+03:00\",\n        \"claim\": \"Source supports the architecture or core technical pattern represented by Wireless Electronic-lock Access Control System.\",\n        \"confidence\": \"HIGH\"\n      },\n      {\n        \"evidenceKey\": \"evidence:security-registry:22d1ab4a2dacc481aa2ed9a4\",\n        \"sourceUrl\": \"https://www.genetec.com/products/unified-security/synergis\",\n        \"sourceType\": \"OFFICIAL_OR_STANDARDS_BODY\",\n        \"observedAt\": \"2026-09-04T23:06:59+03:00\",\n        \"claim\": \"Source supports the architecture or core technical pattern represented by Wireless Electronic-lock Access Control System.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "dfa4a8ecba5695a50cee3900979769434dd91dd66105edeaf7a68425a51724cc", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=RELATION, externalKey=relation:branded_by:70e83a5bd7e7c767e5be) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"RELATION\",\n    \"externalKey\": \"relation:branded_by:70e83a5bd7e7c767e5be\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T22:36:04+03:00\",\n    \"payload\": {\n      \"relationType\": \"BRANDED_BY\",\n      \"sourceExternalKey\": \"model:hikvision:ds-96-recorder-family:ds-9632n-m8\",\n      \"targetExternalKey\": \"brand:hikvision\",\n      \"confidence\": \"HIGH\",\n      \"notes\": \"DS-9632N-M8 is marketed under the Hikvision brand.\"\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:batch003:model:55516d11dd7785af84abc6fa\",\n        \"sourceUrl\": \"https://opensource.hikvision.com/Home/List?id=12&page=1\",\n        \"sourceType\": \"MANUFACTURER_OFFICIAL\",\n        \"observedAt\": \"2026-09-04T22:36:04+03:00\",\n        \"claim\": \"DS-9632N-M8 is marketed under the Hikvision brand.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}]''')
assert len(PROMPTS) == 10, 'expected exactly 10 frozen release items'
assert all(set(p.keys()) == {'item_id','messages'} for p in PROMPTS), 'gold leaked into payload'
print('FINAL V1 RELEASE GATE: frozen items:', len(PROMPTS), '| gold present: False')

WORKING = pathlib.Path('/kaggle/working')
ADAPTER = None
for c in sorted(pathlib.Path('/kaggle/input').rglob('adapter_model.safetensors')):
    if 'checkpoint' not in str(c):
        ADAPTER = c.parent; break
assert ADAPTER is not None, 'adapter not found'
print('adapter dir:', ADAPTER)

t0 = time.time()
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=str(ADAPTER), max_seq_length=3072, load_in_4bit=True, full_finetuning=False)
ADAPTER_LOAD = time.time() - t0
MODEL_LOAD = ADAPTER_LOAD

FastLanguageModel.for_inference(model)
model.eval()
print('device map:', getattr(model, 'hf_device_map', 'n/a'))
print('lm_head device:', next(model.lm_head.parameters()).device if hasattr(model,'lm_head') else 'n/a')

# FIX 1: explicit Harmony terminators + stopping criteria.
TERMINATORS = ['<|return|>', '<|call|>']
STOP_IDS = set()
for tok in TERMINATORS:
    ids = tokenizer.encode(tok, add_special_tokens=False)
    if ids: STOP_IDS.add(ids[-1])
if tokenizer.eos_token_id is not None: STOP_IDS.add(tokenizer.eos_token_id)
print('stop token ids:', sorted(STOP_IDS))
from transformers import StoppingCriteria, StoppingCriteriaList
class HarmonyStop(StoppingCriteria):
    def __init__(self, ids): self.ids = ids
    def __call__(self, input_ids, scores, **kw):
        return all(int(input_ids[i][-1]) in self.ids for i in range(input_ids.shape[0]))

def render(msgs):
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True,
        reasoning_effort='medium', strftime_now=lambda _f: '2026-09-15')

def final_channel(text):
    last = None
    for m in re.finditer(r'<\|channel\|>([A-Za-z_][A-Za-z0-9_]*)\s*<\|message\|>', text):
        s = m.end(); e = len(text)
        for t in ('<|return|>','<|end|>','<|call|>','<|start|>'):
            i = text.find(t, s)
            if i != -1: e = min(e, i)
        if m.group(1) == 'final': last = text[s:e]
    return last

# Warm-up (excluded from the gate; first call pays CUDA/kernel JIT costs).
_ = tokenizer(render(PROMPTS[0]['messages']), return_tensors='pt', add_special_tokens=False).to('cuda')
with torch.inference_mode():
    model.generate(**_ , max_new_tokens=16, do_sample=False,
                   pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id)
print('warm-up done')

rows = []
for i, item in enumerate(PROMPTS):
    t_start = time.time()
    ids = tokenizer(render(item['messages']), return_tensors='pt', add_special_tokens=False).to('cuda')
    prompt_tokens = int(ids['input_ids'].shape[1])
    torch.cuda.synchronize(); t_prefill = time.time()
    with torch.inference_mode():                       # FIX 3
        out = model.generate(**ids, max_new_tokens=3072, do_sample=False,
                             eos_token_id=tokenizer.eos_token_id,
                             pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
                             stopping_criteria=StoppingCriteriaList([HarmonyStop(STOP_IDS)]))
    torch.cuda.synchronize(); t_gen = time.time()
    new_ids = out[0][prompt_tokens:]
    raw = tokenizer.decode(new_ids, skip_special_tokens=False)
    t_ext = time.time()
    ans = final_channel(raw)
    t_end = time.time()
    gen_tokens = int(new_ids.shape[0])
    rows.append({'item_id': item['item_id'], 'prompt_tokens': prompt_tokens,
        'generated_tokens': gen_tokens,
        'prefill_seconds': round(t_prefill - t_start, 3),
        'generation_seconds': round(t_gen - t_prefill, 3),
        'tokens_per_second': round(gen_tokens / max(t_gen - t_prefill, 1e-6), 3),
        'extraction_seconds': round(t_end - t_ext, 4),
        'total_seconds': round(t_end - t_start, 3),
        'termination_reason': 'STOP_TOKEN' if int(new_ids[-1]) in STOP_IDS else 'MAX_NEW_TOKENS',
        'final_channel_found': ans is not None, 'ok': ans is not None,
        'raw': raw, 'prediction': ans})
    r = rows[-1]
    print('ITEM %d/%d id=%s prompt_tok=%d gen_tok=%d prefill=%.2fs gen=%.2fs tok/s=%.2f total=%.2fs term=%s final=%s ok=%s'
          % (i+1, len(PROMPTS), r['item_id'][:12], r['prompt_tokens'], r['generated_tokens'],
             r['prefill_seconds'], r['generation_seconds'], r['tokens_per_second'],
             r['total_seconds'], r['termination_reason'], r['final_channel_found'], r['ok']))
    (WORKING/'predictions.jsonl').write_text(
        '\n'.join(json.dumps(x, ensure_ascii=False) for x in rows)+'\n', encoding='utf-8')

totals = sorted(r['total_seconds'] for r in rows)
median = totals[len(totals)//2]
tps = sum(r['tokens_per_second'] for r in rows)/len(rows)
verdict = 'GREEN' if median <= 180 else ('YELLOW' if median <= 300 else 'RED')
summary = {'MODEL_LOAD_SECONDS': round(MODEL_LOAD,2), 'ADAPTER_LOAD_SECONDS': round(ADAPTER_LOAD,2),
  'GPU_LAYOUT': str(getattr(model,'hf_device_map','n/a')),
  'PEAK_GPU_MEMORY_GiB': round(torch.cuda.max_memory_allocated()/1024**3, 3),
  'rows': rows, 'median_item_seconds': round(median,3),
  'mean_tokens_per_second': round(tps,3), 'final_channel_pass': sum(1 for r in rows if r['final_channel_found']),
  'VERDICT': verdict}
(WORKING/'qualification-summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
(WORKING/'COMPLETED').write_text('GHARIBO-exp-002-qualification' + chr(10), encoding='utf-8')
print(json.dumps({k:v for k,v in summary.items() if k!='rows'}, indent=2))
print('COMPLETE')
